# LeetCode 926 — Flip String to Monotone Increasing bằng Python

Link đề: https://leetcode.com/problems/flip-string-to-monotone-increasing/ · Độ khó: **Medium** · Chủ đề: String (chuỗi), Dynamic Programming (quy hoạch động)

Skeleton LeetCode:

```python
class Solution:
    def minFlipsMonoIncr(self, s: str) -> int:
        ...
```

Ghi chú: chuỗi nhị phân **đơn điệu tăng** có dạng `0...01...1` (một đoạn `0` rồi một đoạn `1`, mỗi đoạn có thể rỗng). Mỗi chuỗi như vậy được xác định bởi **điểm cắt** `i`: `i` ký tự đầu là `0`, phần còn lại là `1`. Số lần lật cho điểm cắt `i` là `(số 1 trong s[:i]) + (số 0 trong s[i:])`. Bài toán quy về chọn điểm cắt tốt nhất, và có thể giải trong **một lượt** với hai biến đếm.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lật một số ký tự (`0 -> 1` hoặc `1 -> 0`) để chuỗi thành dạng `0^a 1^b`, tìm số lần lật ít nhất.
2. Nắm các công cụ Python cần thiết: `str.count`, cắt lát `s[:i]`/`s[i:]`, mảng tổng tiền tố, `itertools.product`, và cập nhật hai biến trong một lượt duyệt.
3. Hiểu nguyên lý: chuỗi đơn điệu tăng ⇔ chọn **điểm cắt**; chi phí = số `1` bên trái (phải lật thành `0`) + số `0` bên phải (phải lật thành `1`).
4. Đi qua 4 cách: thử mọi chuỗi đích (brute), thử mọi điểm cắt bằng tổng tiền tố, quy hoạch động hai trạng thái, và quét một lượt với bộ đếm (cách nộp chính); đối chiếu bằng oracle độc lập.
5. Tránh các lỗi kinh điển: chỉ lật một loại ký tự, tham lam sai, quên hai điểm cắt biên (toàn `0`, toàn `1`), và quên lấy `min` ở bước cập nhật một lượt.

## 2. Đề bài & ví dụ

Một chuỗi nhị phân là **đơn điệu tăng** nếu nó gồm một số ký tự `0` (có thể không có) theo sau là một số ký tự `1` (cũng có thể không có).

Cho chuỗi nhị phân `s`. Bạn có thể **lật** `s[i]`: đổi `0` thành `1` hoặc `1` thành `0`.

Trả về số lần lật **ít nhất** để `s` trở thành đơn điệu tăng.

**Ví dụ 1:** `s = "00110"` -> `1` (lật chữ số cuối để được `00111`).

**Ví dụ 2:** `s = "010110"` -> `2` (được `011111` hoặc `000111`).

**Ví dụ 3:** `s = "00011000"` -> `2` (được `00000000`).

**Ràng buộc:**

- `1 <= s.length <= 10^5`
- `s[i]` là `'0'` hoặc `'1'`.

In [1]:
from typing import List
from itertools import product
import random, time

examples = [
    ("00110", 1),
    ("010110", 2),
    ("00011000", 2),
]
for s, expected in examples:
    print(f"s={s!r} kỳ vọng={expected}")

s='00110' kỳ vọng=1
s='010110' kỳ vọng=2
s='00011000' kỳ vọng=2


## 3. Kiến thức nền

### 3.1. Chuỗi đơn điệu tăng và điểm cắt

Chuỗi đơn điệu tăng độ dài `n` có đúng `n + 1` dạng: `0^i 1^(n-i)` với `i = 0..n`. `i = 0` là toàn `1`, `i = n` là toàn `0`.

In [2]:
n = 4
print([("0" * i + "1" * (n - i)) for i in range(n + 1)])

def is_monotone(t: str) -> bool:
    return all(t[i] <= t[i + 1] for i in range(len(t) - 1))

print(is_monotone("00111"), is_monotone("0101"), is_monotone(""), is_monotone("1"))

['1111', '0111', '0011', '0001', '0000']
True False True True


### 3.2. Chi phí của một điểm cắt

Biến `s` thành `0^i 1^(n-i)`: mỗi `1` trong `s[:i]` phải lật thành `0`, mỗi `0` trong `s[i:]` phải lật thành `1`. Chi phí `cost(i) = count('1' in s[:i]) + count('0' in s[i:])`.

In [3]:
def cost(s: str, i: int) -> int:
    return s[:i].count("1") + s[i:].count("0")

s = "00110"
print([cost(s, i) for i in range(len(s) + 1)], "-> nhỏ nhất", min(cost(s, i) for i in range(len(s) + 1)))

[3, 2, 1, 2, 3, 2] -> nhỏ nhất 1


### 3.3. Tổng tiền tố

`ones[i]` = số ký tự `1` trong `s[:i]`; khi đó số `0` trong `s[i:]` là `(n - i) - (ones[n] - ones[i])`. Mỗi điểm cắt tính trong `O(1)`.

In [4]:
def prefix_ones(s: str) -> List[int]:
    ones = [0]
    for ch in s:
        ones.append(ones[-1] + (ch == "1"))
    return ones

print(prefix_ones("00110"))

[0, 0, 0, 1, 2, 2]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `s = "00110"`, `n = 5`.

| `i` | đích | số `1` trong `s[:i]` | số `0` trong `s[i:]` | chi phí |
|---|---|---|---|---|
| 0 | `11111` | 0 | 3 | 3 |
| 1 | `01111` | 0 | 2 | 2 |
| 2 | `00111` | 0 | 1 | **1** |
| 3 | `00011` | 1 | 1 | 2 |
| 4 | `00001` | 2 | 1 | 3 |
| 5 | `00000` | 2 | 0 | 2 |

Nhỏ nhất là `1` (điểm cắt `i = 2`: đích `00111`).

## 5. Cách 1 — Brute Force: thử mọi chuỗi nhị phân đích

**Ý tưởng.** Liệt kê mọi chuỗi nhị phân `t` độ dài `n` (`2^n`), giữ các chuỗi đơn điệu tăng, đếm số vị trí khác `s` (khoảng cách Hamming) và lấy nhỏ nhất. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [5]:
def flips_brute(s: str) -> int:
    best = None
    for bits in product("01", repeat=len(s)):
        t = "".join(bits)
        if not is_monotone(t):
            continue
        d = sum(a != b for a, b in zip(s, t))
        if best is None or d < best:
            best = d
    return best

for s, expected in examples:
    got = flips_brute(s)
    print(s, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

00110 -> 1 OK
010110 -> 2 OK
00011000 -> 2 OK


**Độ phức tạp và nút thắt.** `O(2^n * n)`: `n = 20` đã `~ 2 * 10^7`. Xem thời gian:

In [6]:
rng = random.Random(926)
for n in (8, 10, 12, 14, 16):
    s = "".join(rng.choice("01") for _ in range(n))
    t0 = time.perf_counter()
    flips_brute(s)
    print(f"n={n:2}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n= 8:       0.2 ms
n=10:       0.7 ms
n=12:       2.3 ms
n=14:       9.6 ms
n=16:      42.9 ms


Thời gian tăng gấp đôi mỗi ký tự. Nút thắt: ta liệt kê cả những chuỗi **không** đơn điệu; chỉ có `n + 1` chuỗi đơn điệu cần xét.

## 6. Ý tưởng thuật toán / bước đột phá

Chỉ có `n + 1` đích đơn điệu tăng (một cho mỗi điểm cắt `i`); chi phí của điểm cắt `i` là `ones_before(i) + zeros_after(i)`. Ta có thể:

- tính mọi chi phí bằng **tổng tiền tố** (`O(n)`);
- hoặc nhận thấy khi dịch điểm cắt từ `i` sang `i + 1`, chi phí chỉ đổi theo ký tự `s[i]` nên cập nhật **một lượt** với hai biến.

| Cách | Ý tưởng |
|---|---|
| 2 | Thử mọi điểm cắt bằng quét `count` (`O(n^2)`) rồi bằng tổng tiền tố (`O(n)`) |
| 3 | Quy hoạch động hai trạng thái: kết thúc bằng `0` / kết thúc bằng `1` |
| 4 | Quét một lượt: `ones` và `ans` (nộp) |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi điểm cắt với tổng tiền tố

**Giải thích từng đoạn.**

- `ones[i]`: số `1` trong `s[:i]`; `total_ones = ones[n]`.
- `zeros_after(i) = (n - i) - (total_ones - ones[i])`.
- `cost(i) = ones[i] + zeros_after(i)`; lấy `min` với `i = 0..n` (gồm cả hai điểm cắt biên).

In [7]:
def flips_prefix(s: str) -> int:
    n = len(s)
    ones = [0] * (n + 1)
    for i, ch in enumerate(s):
        ones[i + 1] = ones[i] + (ch == "1")
    total = ones[n]
    return min(ones[i] + (n - i) - (total - ones[i]) for i in range(n + 1))

for s, expected in examples:
    got = flips_prefix(s)
    print(s, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

00110 -> 1 OK
010110 -> 2 OK
00011000 -> 2 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def minFlipsMonoIncr(self, s: str) -> int:
        n = len(s)
        ones = [0] * (n + 1)
        for i, ch in enumerate(s):
            ones[i + 1] = ones[i] + (ch == "1")
        total = ones[n]
        return min(ones[i] + (n - i) - (total - ones[i]) for i in range(n + 1))

sol_prefix = Solution().minFlipsMonoIncr
assert [sol_prefix(s) for s, _ in examples] == [e for _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n)` thời gian, `O(n)` bộ nhớ.

---

### 7.2. Cách 3 — Quy hoạch động hai trạng thái

**Ý tưởng khác.** Duyệt từng ký tự và giữ hai giá trị cho tiền tố đã xử lý:

- `dp0`: số lần lật ít nhất để tiền tố **toàn `0`**;
- `dp1`: số lần lật ít nhất để tiền tố là **đơn điệu tăng và kết thúc bằng `1`** (hoặc tiền tố đơn điệu bất kỳ có ký tự cuối là `1`).

Với ký tự `c`:

- `new_dp0 = dp0 + (c == "1")` (muốn giữ toàn `0` thì `1` phải lật);
- `new_dp1 = min(dp0, dp1) + (c == "0")` (ký tự này là `1`; trước đó là toàn `0` hoặc đã có `1`; nếu `c == "0"` phải lật).

Đáp án `min(dp0, dp1)`.

**Giải thích từng đoạn.** Khởi tạo `dp0 = 0`, `dp1 = 0` (tiền tố rỗng; `dp1` cho tiền tố rỗng chưa kết thúc bằng `1` nhưng việc dùng `min(dp0, dp1)` ở bước sau vẫn đúng vì tiền tố rỗng hợp lệ).

In [9]:
def flips_dp(s: str) -> int:
    dp0 = dp1 = 0
    for c in s:
        dp0, dp1 = dp0 + (c == "1"), min(dp0, dp1) + (c == "0")
    return min(dp0, dp1)

for s, expected in examples:
    got = flips_dp(s)
    print(s, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

00110 -> 1 OK
010110 -> 2 OK
00011000 -> 2 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def minFlipsMonoIncr(self, s: str) -> int:
        dp0 = dp1 = 0
        for c in s:
            dp0, dp1 = dp0 + (c == "1"), min(dp0, dp1) + (c == "0")
        return min(dp0, dp1)

sol_dp = Solution().minFlipsMonoIncr
assert [sol_dp(s) for s, _ in examples] == [e for _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n)` thời gian, `O(1)` bộ nhớ.

---

### 7.3. Cách 4 — Quét một lượt với bộ đếm (cách nộp chính)

**Ý tưởng khác.** Giữ `ones` = số `1` đã gặp và `ans` = số lần lật ít nhất để tiền tố đã duyệt thành đơn điệu tăng. Với ký tự mới:

- nếu là `'1'`: thêm vào đuôi không phá tính đơn điệu: `ones += 1`, `ans` giữ nguyên;
- nếu là `'0'`: có hai lựa chọn: **lật nó thành `1`** (`ans + 1`) hoặc **lật mọi `1` trước đó thành `0`** (`ones`): `ans = min(ans + 1, ones)`.

**Giải thích từng đoạn.** `ans` luôn bằng chi phí tối ưu của tiền tố hiện tại. Khi gặp `0`: nếu giữ lời giải tối ưu trước đó và lật `0` này thành `1` thì tốn thêm `1`; hoặc biến cả tiền tố thành toàn `0` bằng cách lật mọi `1` (tốn `ones`).

In [11]:
def flips_one_pass(s: str) -> int:
    ones = ans = 0
    for c in s:
        if c == "1":
            ones += 1
        else:
            ans = min(ans + 1, ones)
    return ans

for s, expected in examples:
    got = flips_one_pass(s)
    print(s, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

00110 -> 1 OK
010110 -> 2 OK
00011000 -> 2 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def minFlipsMonoIncr(self, s: str) -> int:
        ones = ans = 0
        for c in s:
            if c == "1":
                ones += 1
            else:
                ans = min(ans + 1, ones)
        return ans

sol_one = Solution().minFlipsMonoIncr
assert [sol_one(s) for s, _ in examples] == [e for _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n)` thời gian, `O(1)` bộ nhớ phụ.

## 8. TRACE từng bước

### 8.1. Cách 4 cho ví dụ 2: `s = "010110"`

In [13]:
def trace_one_pass(s):
    ones = ans = 0
    print(f"{'ký tự':6} {'ones':5} {'ans':4} ghi chú")
    for c in s:
        if c == "1":
            ones += 1
            note = "gặp 1: ones tăng"
        else:
            old = ans
            ans = min(ans + 1, ones)
            note = f"gặp 0: min(ans + 1 = {old + 1}, ones = {ones}) = {ans}"
        print(f"{c:6} {ones:5} {ans:4} {note}")
    return ans

assert trace_one_pass("010110") == 2

ký tự  ones  ans  ghi chú
0          0    0 gặp 0: min(ans + 1 = 1, ones = 0) = 0
1          1    0 gặp 1: ones tăng
0          1    1 gặp 0: min(ans + 1 = 1, ones = 1) = 1
1          2    1 gặp 1: ones tăng
1          3    1 gặp 1: ones tăng
0          3    2 gặp 0: min(ans + 1 = 2, ones = 3) = 2


### 8.2. Cách 3 (DP hai trạng thái) cho `s = "00110"`

In [14]:
def trace_dp(s):
    dp0 = dp1 = 0
    print(f"khởi tạo dp0={dp0}, dp1={dp1}")
    for c in s:
        n0, n1 = dp0 + (c == "1"), min(dp0, dp1) + (c == "0")
        print(f"ký tự {c}: dp0 = {dp0} + {int(c == '1')} = {n0}; dp1 = min({dp0}, {dp1}) + {int(c == '0')} = {n1}")
        dp0, dp1 = n0, n1
    return min(dp0, dp1)

assert trace_dp("00110") == 1

khởi tạo dp0=0, dp1=0
ký tự 0: dp0 = 0 + 0 = 0; dp1 = min(0, 0) + 1 = 1
ký tự 0: dp0 = 0 + 0 = 0; dp1 = min(0, 1) + 1 = 1
ký tự 1: dp0 = 0 + 1 = 1; dp1 = min(0, 1) + 0 = 0
ký tự 1: dp0 = 1 + 1 = 2; dp1 = min(1, 0) + 0 = 0
ký tự 0: dp0 = 2 + 0 = 2; dp1 = min(2, 0) + 1 = 1


### 8.3. Cách 2 (mọi điểm cắt) cho `s = "00011000"`

`ones = [0,0,0,0,1,2,2,2,2]`; chi phí `i`: `i=0`: `0 + 6 = 6`; ... `i = 8`: `ones[8] = 2` (toàn `0`): `2`; điểm cắt `i = 3` (`00011111`...): hãy để máy tính.

In [15]:
s = "00011000"
ones = prefix_ones(s)
print([cost(s, i) for i in range(len(s) + 1)])
assert min(cost(s, i) for i in range(len(s) + 1)) == 2 == Solution().minFlipsMonoIncr(s)

[6, 5, 4, 3, 4, 5, 4, 3, 2]


## 9. Code nộp LeetCode

Ta chọn **Cách 4** (quét một lượt với bộ đếm): `O(n)` thời gian, `O(1)` bộ nhớ, rất ngắn.

```python
class Solution:
    def minFlipsMonoIncr(self, s: str) -> int:
        ones = ans = 0
        for c in s:
            if c == "1":
                ones += 1
            else:
                ans = min(ans + 1, ones)
        return ans
```

In [16]:
class Solution:
    def minFlipsMonoIncr(self, s: str) -> int:
        ones = ans = 0
        for c in s:
            if c == "1":
                ones += 1
            else:
                ans = min(ans + 1, ones)
        return ans

sol = Solution()
for s, expected in examples:
    got = sol.minFlipsMonoIncr(s)
    print(s, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

00110 -> 1 OK
010110 -> 2 OK
00011000 -> 2 OK


## 10. Vì sao đúng

**(a) Đích đơn điệu tăng ứng với điểm cắt.** Chuỗi đơn điệu tăng độ dài `n` là `0^i 1^(n-i)` với `0 <= i <= n`; lật `s` thành chuỗi này tốn đúng số vị trí khác nhau: `1` trong `s[:i]` (phải thành `0`) và `0` trong `s[i:]` (phải thành `1`). Đáp án là `min` theo `i` (Cách 2 tính trực tiếp).

**(b) Cách 4 (quét một lượt).** Gọi `f(k)` là số lần lật ít nhất để tiền tố `s[:k]` thành đơn điệu tăng, `ones_k` là số `1` trong `s[:k]`. Bất biến: sau `k` ký tự, `ans = f(k)`, `ones = ones_k`.
   - Ký tự `'1'`: gắn `1` vào đuôi một chuỗi đích đơn điệu tăng của `s[:k]` luôn đơn điệu, nên `f(k+1) = f(k)`.
   - Ký tự `'0'`: xét đích tối ưu của `s[:k+1]`. Nếu ký tự cuối của đích là `1` thì cả đích là `... 1` mà vị trí cuối của `s` là `0` nên phải lật; phần đầu là đích đơn điệu của `s[:k]`: chi phí `>= f(k) + 1`, và đạt được. Nếu ký tự cuối của đích là `0` thì đích toàn `0` (đơn điệu tăng kết thúc bằng `0` phải toàn `0`): chi phí đúng `ones_{k+1} = ones_k` (ký tự cuối `0` không cần lật). Vậy `f(k+1) = min(f(k) + 1, ones_k)`.

**(c) Cách 3 (DP).** `dp0(k)` = chi phí để `s[:k]` thành toàn `0` = `ones_k`; `dp1(k)` = chi phí tối thiểu để `s[:k]` thành đơn điệu tăng với ký tự cuối là `1` (bắt buộc `k >= 1`). Chuyển: ký tự `k+1` đặt `1`: tiền tố đứng trước có thể toàn `0` hoặc đơn điệu bất kỳ kết thúc bằng `1`: `min(dp0, dp1) + [s[k] == '0']`. Đáp án `min(dp0, dp1)` ứng với mọi đích (kết thúc bằng `0` thì toàn `0`).

**Kiểm chứng bằng máy.** Oracle **độc lập**: vét cạn mọi chuỗi đích (`n <= 14`) và, cho `n` lớn, **thử mọi điểm cắt bằng đếm trực tiếp** `s[:i].count('1') + s[i:].count('0')` (`O(n^2)`), khác hẳn cả ba cách `O(n)`.

In [17]:
def oracle_split_scan(s: str) -> int:
    return min(s[:i].count("1") + s[i:].count("0") for i in range(len(s) + 1))

rng = random.Random(1926)
for _ in range(500):
    n = rng.randint(1, 12)
    s = "".join(rng.choice("01") for _ in range(n))
    assert flips_brute(s) == oracle_split_scan(s), s
print("vét cạn mọi chuỗi đích và thử mọi điểm cắt khớp nhau trên 500 chuỗi ngẫu nhiên")

vét cạn mọi chuỗi đích và thử mọi điểm cắt khớp nhau trên 500 chuỗi ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi chuỗi đích | `O(2^n * n)` | `O(n)` |
| 2. Mọi điểm cắt (tiền tố) | `O(n)` | `O(n)` |
| 3. DP hai trạng thái | `O(n)` | `O(1)` |
| 4. Quét một lượt | `O(n)` | `O(1)` |

Đo thử với `n = 10^5` (cỡ tối đa) và so với bản `O(n^2)`:

In [18]:
rng = random.Random(11)
n = 10 ** 5
s = "".join(rng.choice("01") for _ in range(n))
ref = sol_prefix(s)
for name, fn in (("Cách 2 tiền tố", sol_prefix), ("Cách 3 DP", sol_dp), ("Cách 4 một lượt", sol_one), ("nộp", Solution().minFlipsMonoIncr)):
    t0 = time.perf_counter()
    got = fn(s)
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:16} n={n}: {dt:8.2f} ms (đáp án {got})")
for m in (500, 1000, 2000):
    t0 = time.perf_counter()
    oracle_split_scan(s[:m])
    print(f"mọi điểm cắt bằng count, n={m}: {1000 * (time.perf_counter() - t0):7.2f} ms")

Cách 2 tiền tố   n=100000:    18.58 ms (đáp án 49781)
Cách 3 DP        n=100000:     8.77 ms (đáp án 49781)
Cách 4 một lượt  n=100000:     5.16 ms (đáp án 49781)
nộp              n=100000:     4.96 ms (đáp án 49781)
mọi điểm cắt bằng count, n=500:    0.36 ms
mọi điểm cắt bằng count, n=1000:    1.24 ms
mọi điểm cắt bằng count, n=2000:    4.68 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [19]:
def solutions_list():
    return {
        "prefix": flips_prefix,
        "dp": flips_dp,
        "one_pass": flips_one_pass,
        "sol_prefix": sol_prefix,
        "sol_dp": sol_dp,
        "sol_one": sol_one,
        "final": Solution().minFlipsMonoIncr,
    }

def run_all(s, expected=None, brute_limit=14):
    ref = flips_brute(s) if len(s) <= brute_limit else oracle_split_scan(s)
    for name, fn in solutions_list().items():
        assert fn(s) == ref, (name, s)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", s, expected, ref)
    return ref

# 12.1 ví dụ đề
for s, expected in examples:
    assert run_all(s, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ("0", 0),                              # một ký tự 0
    ("1", 0),                              # một ký tự 1
    ("01", 0),                             # đã đơn điệu
    ("10", 1),                             # đảo: lật một ký tự
    ("0" * 10, 0),                         # toàn 0
    ("1" * 10, 0),                         # toàn 1
    ("0" * 5 + "1" * 5, 0),                # đã đơn điệu
    ("1" * 5 + "0" * 5, 5),                # đảo hoàn toàn: lật nửa
    ("1010", 2),                           # xen kẽ: 0011 hoặc 0000, ... cần 2
    ("0101", 1),                           # 0111 (lật một)
    ("10011", 1),                          # lật 1 đầu thành 0: 00011
    ("11110", 1),                          # lật 0 cuối
    ("01" * 8, 7),                         # xen kẽ độ dài 16: bằng oracle
]
for s, expected in edge:
    run_all(s, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 13 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_min_count(s):                           # chỉ lật một loại ký tự: min(số 1, số 0)
    return min(s.count("1"), s.count("0"))

def wrong_flip_zeros_after_first_one(s):          # lật mọi 0 sau số 1 đầu tiên
    i = s.find("1")
    return 0 if i == -1 else s[i:].count("0")

def wrong_no_edges(s):                            # bỏ hai điểm cắt biên (toàn 0, toàn 1)
    n = len(s)
    if n < 2:
        return 0
    return min(s[:i].count("1") + s[i:].count("0") for i in range(1, n))

def wrong_no_min(s):                              # cập nhật một lượt nhưng quên min
    ones = ans = 0
    for c in s:
        if c == "1":
            ones += 1
        else:
            ans += 1
    return ans

def wrong_swap(s):                                # nhầm: ans = max(ans + 1, ones)
    ones = ans = 0
    for c in s:
        if c == "1":
            ones += 1
        else:
            ans = max(ans + 1, ones)
    return ans

def find_counter(wrong, tries=20000, seed=3):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 10)
        s = "".join(rng.choice("01") for _ in range(n))
        if wrong(s) != oracle_split_scan(s):
            return s
    return None

counters = {}
for name, fn in (("chỉ lật một loại ký tự", wrong_min_count), ("lật mọi 0 sau số 1 đầu", wrong_flip_zeros_after_first_one),
                 ("bỏ điểm cắt biên", wrong_no_edges), ("quên min khi gặp 0", wrong_no_min), ("dùng max thay min", wrong_swap)):
    s = find_counter(fn)
    assert s is not None, name
    counters[name] = s
    print(f"{name:28} s={s!r}: sai -> {fn(s)}, đúng -> {oracle_split_scan(s)}")
    run_all(s)
print("12.3 phản ví dụ: OK")

chỉ lật một loại ký tự       s='0110': sai -> 2, đúng -> 1
lật mọi 0 sau số 1 đầu       s='100': sai -> 2, đúng -> 1
bỏ điểm cắt biên             s='100': sai -> 2, đúng -> 1
quên min khi gặp 0           s='0110': sai -> 2, đúng -> 1
dùng max thay min            s='0110': sai -> 2, đúng -> 1
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: mọi cách so với vét cạn (n <= 14)
rng = random.Random(4926)
for _ in range(1500):
    n = rng.randint(1, 14)
    p = rng.choice([0.1, 0.5, 0.9])
    s = "".join("1" if rng.random() < p else "0" for _ in range(n))
    run_all(s)
print("12.4 ngẫu nhiên n <= 14 (1500 ca): OK")

# 12.5 trung bình: n tới 300 so với oracle thử mọi điểm cắt
rng = random.Random(5926)
for _ in range(300):
    n = rng.randint(15, 300)
    p = rng.choice([0.05, 0.5, 0.95])
    s = "".join("1" if rng.random() < p else "0" for _ in range(n))
    run_all(s)
print("12.5 ngẫu nhiên n tới 300 (300 ca): OK")

# 12.6 lớn n = 10^5 nhiều dạng; so các cách O(n) với nhau và với oracle trên tiền tố 3000 ký tự
rng = random.Random(6926)
n = 10 ** 5
big = {
    "ngẫu nhiên": "".join(rng.choice("01") for _ in range(n)),
    "toàn 0": "0" * n,
    "toàn 1": "1" * n,
    "nửa 1 nửa 0": "1" * (n // 2) + "0" * (n // 2),
    "xen kẽ": "01" * (n // 2),
    "đã đơn điệu": "0" * (n // 3) + "1" * (n - n // 3),
}
expected_big = {"toàn 0": 0, "toàn 1": 0, "nửa 1 nửa 0": n // 2, "đã đơn điệu": 0}
for name, s in big.items():
    vals = {fname: fn(s) for fname, fn in solutions_list().items()}
    assert len(set(vals.values())) == 1, (name, vals)
    if name in expected_big:
        assert vals["final"] == expected_big[name], name
    print(f"  {name:14} đáp án {vals['final']}")
print("12.6 n = 10^5: OK")

# 12.7 tính chất: đáp án <= min(số 1, số 0); thêm "0" vào đầu không đổi, thêm "1" vào cuối không đổi; đảo + đối xứng
rng = random.Random(7926)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 30)))
    f = Solution().minFlipsMonoIncr
    assert f(s) <= min(s.count("1"), s.count("0"))
    assert f("0" + s) == f(s) and f(s + "1") == f(s)
    # đối xứng: đảo chuỗi và đổi 0<->1 cho cùng đáp án
    mirrored = "".join("1" if c == "0" else "0" for c in reversed(s))
    assert f(mirrored) == f(s)
print("12.7 tính chất (<= min, thêm 0 đầu/1 cuối, đối xứng): OK")

12.4 ngẫu nhiên n <= 14 (1500 ca): OK
12.5 ngẫu nhiên n tới 300 (300 ca): OK
  ngẫu nhiên     đáp án 49694
  toàn 0         đáp án 0


  toàn 1         đáp án 0
  nửa 1 nửa 0    đáp án 50000
  xen kẽ         đáp án 49999
  đã đơn điệu    đáp án 0
12.6 n = 10^5: OK
12.7 tính chất (<= min, thêm 0 đầu/1 cuối, đối xứng): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lật một loại ký tự (`min(số 1, số 0)`)

Nghĩ rằng chỉ cần "lật hết `1` thành `0`" hoặc "lật hết `0` thành `1`". Nhưng có thể lật **cả hai loại** (đích `0^i 1^(n-i)` với `0 < i < n`).

In [22]:
s = counters["chỉ lật một loại ký tự"]
print("s =", s, "| sai ->", wrong_min_count(s), "| đúng ->", Solution().minFlipsMonoIncr(s))
assert wrong_min_count(s) != Solution().minFlipsMonoIncr(s)

s = 0110 | sai -> 2 | đúng -> 1


Vì sao sai: đích tối ưu có thể ở giữa. **Đúng:** xét mọi điểm cắt.

### Lỗi 2 — Lật mọi `0` đứng sau số `1` đầu tiên

Tham lam này bỏ qua phương án lật các `1` thành `0`.

In [23]:
s = counters["lật mọi 0 sau số 1 đầu"]
print("s =", s, "| sai ->", wrong_flip_zeros_after_first_one(s), "| đúng ->", Solution().minFlipsMonoIncr(s))
assert wrong_flip_zeros_after_first_one(s) != Solution().minFlipsMonoIncr(s)

s = 100 | sai -> 2 | đúng -> 1


Vì sao sai: khi có nhiều `1` rồi ít `0` thì lật `0` rẻ hơn, nhưng với `"100000"` lật chữ `1` duy nhất rẻ hơn lật năm chữ `0`. **Đúng:** so sánh hai phương án ở mỗi `0` (hoặc xét mọi điểm cắt).

### Lỗi 3 — Quên hai điểm cắt biên

Đích toàn `0` (`i = n`) và toàn `1` (`i = 0`) cũng hợp lệ.

In [24]:
s = counters["bỏ điểm cắt biên"]
print("s =", s, "| sai ->", wrong_no_edges(s), "| đúng ->", Solution().minFlipsMonoIncr(s))
assert wrong_no_edges(s) != Solution().minFlipsMonoIncr(s)

s = 100 | sai -> 2 | đúng -> 1


Vì sao sai: ví dụ 3 (`"00011000"`) có đáp án `2` đạt ở đích toàn `0`. **Đúng:** `range(n + 1)`.

### Lỗi 4 — Quên `min` khi gặp `0` (một lượt)

`ans += 1` mỗi khi gặp `0` bỏ qua phương án lật mọi `1` trước đó.

In [25]:
s = counters["quên min khi gặp 0"]
print("s =", s, "| sai ->", wrong_no_min(s), "| đúng ->", Solution().minFlipsMonoIncr(s))
assert wrong_no_min(s) != Solution().minFlipsMonoIncr(s)
s = counters["dùng max thay min"]
print("dùng max: s =", s, "| sai ->", wrong_swap(s), "| đúng ->", Solution().minFlipsMonoIncr(s))
assert wrong_swap(s) != Solution().minFlipsMonoIncr(s)

s = 0110 | sai -> 2 | đúng -> 1
dùng max: s = 0110 | sai -> 2 | đúng -> 1


Vì sao sai: với `"1111110"` phương án rẻ nhất là lật một `0`; với `"1000000"`... cần so sánh `ans + 1` và `ones`. **Đúng:** `ans = min(ans + 1, ones)`.

### Lỗi 5 — Thử mọi điểm cắt bằng `count` (`O(n^2)`)

Đúng nhưng với `n = 10^5` là `10^10` thao tác (xem mục 11). **Đúng:** tổng tiền tố hoặc quét một lượt.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "biến chuỗi nhị phân thành dạng `0...01...1` với ít lật nhất" ⇒ **điểm cắt** (`ones_before + zeros_after`) hoặc **DP hai trạng thái**.
- **Quét một lượt:** `ones` và `ans`; gặp `0`: `ans = min(ans + 1, ones)`.
- Hai điểm cắt biên (toàn `0`, toàn `1`) luôn là ứng viên.
- Cùng họ: Minimum Number of Flips to Make the Binary String Alternating (1888), Max Consecutive Ones III (1004), Minimum Swaps to Group All 1's Together (1151), Number of Substrings With Only 1s.
- Test nên có: `"0"`, `"1"`, `"10"`, toàn `0`/toàn `1`, `"1"*k + "0"*k`, xen kẽ, `n = 10^5`.

## 15. Biến thể

1. **Đơn điệu giảm** (`1...10...0`): đối xứng.
2. **Đơn điệu theo một trong hai chiều** (tăng hoặc giảm): `min` của hai đáp án.
3. **Trả về chuỗi đích** (bài tập 5).
4. Bài liên quan: 1888.

Dưới đây là biến thể 2 kiểm bằng vét cạn.

In [26]:
def flips_monotone_either(s: str) -> int:
    inc = Solution().minFlipsMonoIncr(s)
    dec = Solution().minFlipsMonoIncr(s[::-1])               # đảo chuỗi: giảm trở thành tăng
    return min(inc, dec)

def either_brute(s):
    best = None
    for bits in product("01", repeat=len(s)):
        t = "".join(bits)
        if is_monotone(t) or is_monotone(t[::-1]):
            d = sum(a != b for a, b in zip(s, t))
            best = d if best is None or d < best else best
    return best

rng = random.Random(15)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 10)))
    assert flips_monotone_either(s) == either_brute(s), s
print("biến thể 2 khớp vét cạn; ví dụ:", flips_monotone_either("1100"), flips_monotone_either("0110"))

biến thể 2 khớp vét cạn; ví dụ: 0 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi đích | Định nghĩa | `O(2^n n)` | `O(n)` | Oracle, `n <= 14` |
| 2. Điểm cắt + tiền tố | `ones_before + zeros_after` | `O(n)` | `O(n)` | Dễ hiểu |
| 3. DP hai trạng thái | Kết thúc bằng 0 / 1 | `O(n)` | `O(1)` | Dễ mở rộng (chi phí khác nhau) |
| 4. Quét một lượt | `ans = min(ans + 1, ones)` | `O(n)` | `O(1)` | **Cách nộp chính** |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(prefix_ones("00110"))
print(cost("00110", 3))
print(is_monotone("0011"), is_monotone("0110"))
```

In [27]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s = "10011"`: ghi số lần lật tối thiểu và một chuỗi đích.

In [28]:
# Bài 2: điền kết quả tính tay
hand_answer = None
hand_target = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành thử mọi điểm cắt bằng tổng tiền tố.

In [29]:
def exercise_3_prefix(s: str) -> int:
    n = len(s)
    ones = [0] * (n + 1)
    for i, ch in enumerate(s):
        # TODO 1: ones[i + 1] = số '1' trong s[:i + 1]
        pass
    total = ones[n]
    # TODO 2: trả về min theo i = 0..n của (số 1 trong s[:i]) + (số 0 trong s[i:])
    return None


# for s, expected in examples:
#     assert exercise_3_prefix(s) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_dp(s)` bằng DP hai trạng thái `dp0`, `dp1`.

In [30]:
def exercise_4_dp(s: str) -> int:
    # TODO
    pass


# for s, expected in examples:
#     assert exercise_4_dp(s) == expected

### Bài 5 — Mini challenge: trả về một chuỗi đích tối ưu (khó)

Viết `exercise_5_target(s)` trả về một chuỗi đơn điệu tăng `t` cùng độ dài `s` sao cho số vị trí khác `s` bằng đáp án tối thiểu. Có nhiều đáp án; ta kiểm tra bằng hàm kiểm tra.

```python
exercise_5_target("00110")   -> "00111"
exercise_5_target("010110")  -> "011111" hoặc "000111"
```

In [31]:
def exercise_5_target(s: str) -> str:
    # TODO
    pass


# for s, expected in examples:
#     t = exercise_5_target(s)
#     assert len(t) == len(s) and is_monotone(t) and sum(a != b for a, b in zip(s, t)) == expected

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `prefix_ones` cho số `1` trong từng tiền tố.
- Mức 2: `cost(s, 3)` = số `1` trong `"001"` cộng số `0` trong `"10"`.
- Mức 3: `[0, 0, 0, 1, 2, 2]`, `2`, `True False`.

**Bài 2**
- Mức 1: thử đích `00011`.
- Mức 2: `10011` so với `00011` khác ở vị trí `0`.
- Mức 3: đáp án `1`, đích `00011`.

**Bài 3**
- Mức 1: `ones[i + 1] = ones[i] + (ch == "1")`.
- Mức 2: số `0` trong `s[i:]` là `(n - i) - (total - ones[i])`.
- Mức 3: `return min(ones[i] + (n - i) - (total - ones[i]) for i in range(n + 1))`.

**Bài 4**
- Mức 1: `dp0` là chi phí toàn `0`, `dp1` là chi phí kết thúc bằng `1`.
- Mức 2: `dp0 += (c == "1")`, `dp1 = min(dp0_cũ, dp1) + (c == "0")`.
- Mức 3: cập nhật đồng thời hai biến; trả `min(dp0, dp1)`.

**Bài 5**
- Mức 1: tìm điểm cắt `i` tối thiểu của `ones_before + zeros_after`.
- Mức 2: đích là `"0" * i + "1" * (n - i)`.
- Mức 3: trả đích với `i` tốt nhất.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [32]:
print(prefix_ones("00110"))
print(cost("00110", 3))
print(is_monotone("0011"), is_monotone("0110"))

assert prefix_ones("00110") == [0, 0, 0, 1, 2, 2]
assert cost("00110", 3) == 2
assert (is_monotone("0011"), is_monotone("0110")) == (True, False)

[0, 0, 0, 1, 2, 2]
2
True False


### Đáp án Bài 2

Đích `00011`: chỉ khác `10011` ở vị trí `0`: `1` lần lật.

In [33]:
hand_answer_solution = 1
hand_target_solution = "00011"
s = "10011"
assert Solution().minFlipsMonoIncr(s) == hand_answer_solution == flips_brute(s)
assert is_monotone(hand_target_solution) and sum(a != b for a, b in zip(s, hand_target_solution)) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [34]:
def exercise_3_prefix_solution(s: str) -> int:
    n = len(s)
    ones = [0] * (n + 1)
    for i, ch in enumerate(s):
        ones[i + 1] = ones[i] + (ch == "1")
    total = ones[n]
    return min(ones[i] + (n - i) - (total - ones[i]) for i in range(n + 1))

for s, expected in examples:
    assert exercise_3_prefix_solution(s) == expected
rng = random.Random(3)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 12)))
    assert exercise_3_prefix_solution(s) == flips_brute(s)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [35]:
def exercise_4_dp_solution(s: str) -> int:
    dp0 = dp1 = 0
    for c in s:
        dp0, dp1 = dp0 + (c == "1"), min(dp0, dp1) + (c == "0")
    return min(dp0, dp1)

for s, expected in examples:
    assert exercise_4_dp_solution(s) == expected
rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 12)))
    assert exercise_4_dp_solution(s) == flips_brute(s)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [36]:
def exercise_5_target_solution(s: str) -> str:
    n = len(s)
    best_i = min(range(n + 1), key=lambda i: s[:i].count("1") + s[i:].count("0"))
    return "0" * best_i + "1" * (n - best_i)

def check_target(s, t):
    return len(t) == len(s) and is_monotone(t) and sum(a != b for a, b in zip(s, t)) == flips_brute(s)

for s, expected in examples:
    t = exercise_5_target_solution(s)
    assert len(t) == len(s) and is_monotone(t) and sum(a != b for a, b in zip(s, t)) == expected
rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 12)))
    assert check_target(s, exercise_5_target_solution(s)), s
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (quét một lượt)

```python
class Solution:
    def minFlipsMonoIncr(self, s: str) -> int:
        ones = ans = 0                       # ones: số '1' đã gặp; ans: lật ít nhất cho tiền tố
        for c in s:
            if c == "1":
                ones += 1                    # gắn '1' vào đuôi: không phá tính đơn điệu
            else:
                ans = min(ans + 1, ones)     # lật '0' này thành '1'  HOẶC  lật mọi '1' trước đó thành '0'
        return ans
```

Công thức nhớ nhanh: đích `0^i 1^(n-i)`; chi phí `i` = `ones_before(i) + zeros_after(i)`; gặp `0`: `ans = min(ans + 1, ones)`.

### Checklist trước khi submit

- [ ] Xét cả hai loại lật (không chỉ `min(số 1, số 0)`)?
- [ ] Hai điểm cắt biên (toàn `0`, toàn `1`) được tính?
- [ ] Gặp `0`: `ans = min(ans + 1, ones)` (có `min`, không `max`)?
- [ ] `O(n)`, không dùng `count` trong vòng lặp?
- [ ] Đã thử: `"0"`, `"1"`, `"10"`, toàn `0`/`1`, `"1"*k + "0"*k`, xen kẽ, `n = 10^5`?
- [ ] Tên method đúng `minFlipsMonoIncr(self, s)` và trả `int`?